# Datamine ISOTRI Process Example

This notebook demonstrates how to configure and run the **`isotri`** process wrapper in `dmstudio`.

## Process Description

## ISOTRI

Generate a plot file of an isometric view of a triangulated (wireframe) model, at any desired orientation.

* Scaling is completely automatic in this process, and any plot file or null prototype may be used; all that need be defined is the paper size.

* The hidden line removal algorithm can be slow when using large data sets, so it is recommended that an initial plot with @**HIDDEN** set to 1 be produced to check the view parameters.

### Input Files:

* **proto** (Plot Prototype File):
Prototype plot file.
Required=Yes

* **wiretr** (Wireframe Triangle):
Input wireframe triangle file .
Required=Yes

* **wirept** (Wireframe Points):
Input wireframe point file.
Required=Yes

### Output Files:

* **plot** (Plot):
Output plot file.
Required=Yes

### Fields:

### Parameters:

* **pxmin**:
Minimum data X value to be plotted.
Range=Undefined
Values=Undefined
Default=Undefined
Required=Yes

* **pxmax**:
Maximum data X value to be plotted.
Range=Undefined
Values=Undefined
Default=Undefined
Required=Yes

* **pymin**:
Minimum data Y value to be plotted.
Range=Undefined
Values=Undefined
Default=Undefined
Required=Yes

* **pymax**:
Maximum data Y value to be plotted.
Range=Undefined
Values=Undefined
Default=Undefined
Required=Yes

* **pzmin**:
Minimum data Z value to be plotted.
Range=Undefined
Values=Undefined
Default=Undefined
Required=Yes

* **pzmax**:
Maximum data Z value to be plotted.
Range=Undefined
Values=Undefined
Default=Undefined
Required=Yes

* **rotate**:
The rotation angle of the direction of view, in degrees horizontally and clockwise from
the data Y axis. Default (45). E.g. if model Y and X axes are parallel to North 45 =
Looking North-East 225 = Looking South-West
Range=0,360
Values=Undefined
Default=45
Required=No

* **elevate**:
The rotation angle of the direction of view, in degrees vertically from data X-Y plane.
Default (30). E.g. if model Y and X axes are parallel to North 0 = Looking horizontally,
along X-Y plane. +90 = Looking vertically downwards -90 = Looking vertically upwards
Range=-90,90
Values=Undefined
Default=30
Required=No

* **hidden**:
Control of hidden line display. Options: 0: Hidden lines are NOT displayed.; 1: Hidden
lines are displayed.
Range=0,1
Values=0,1
Default=0
Required=No

* **charsize**:
Character size in millimetres.
Range=
Values=
Default=3
Required=No

* **aspratio**:
Aspect ratio, width / ht. for chars.
Range=
Values=
Default=0.9
Required=No

* **append**:
Plot append flag. If set to 1 then the new plot will be appended to the PLOT file, if
it exists and is a valid plot file. N.B. Scaling is fully automatic in this process.
Range=
Values=
Default=0
Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('isotri')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute isotri
print("Running isotri...")
dm_cmd.isotri(
    proto_i='t_mod1',  # required
    wiretr_i='t_SurfaceTriangles',  # required
    wirept_i='t_SurfaceTriangles',  # required
    plot_o='t_isotri_out',  # required
    pxmin_p='required_val',  # required
    pxmax_p='required_val',  # required
    pymin_p='required_val',  # required
    pymax_p='required_val',  # required
    pzmin_p='required_val',  # required
    pzmax_p='required_val',  # required
    # rotate_p=45,  # optional
    # elevate_p=30,  # optional
    # hidden_p=0,  # optional
    # charsize_p=3,  # optional
    # aspratio_p=0.9,  # optional
    # append_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("isotri execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_isotri_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")